# AE2 - Ideacion, diseno y documentacion de la necesidad
## Proyecto integrador: Caracterizacion de la cancelacion de clientes en una empresa de telecomunicaciones

**Autor:** Diego Mauricio Bedoya Valencia - Institucion Universitaria Digital de Antioquia

Este notebook corresponde a la **etapa 1** del proyecto integrador. Su proposito es **conocer los datos** para formular la necesidad analitica; no realiza un analisis detallado.

Flujo:

```
Hugging Face -> HFDatasetLoader -> DataFrame -> descripcion basica -> DatasetSummarizer -> resultados/resumen_dataframe.txt
```

| Dato | Valor |
|---|---|
| Nombre | Telco Customer Churn (IBM Sample Dataset) |
| Identificador | `scikit-learn/churn-prediction` |
| Enlace | https://huggingface.co/datasets/scikit-learn/churn-prediction |
| Configuracion | `default` (unica) |
| Split | `train` (unico split publicado) |
| Revision (commit) | `aa09900373d90780ee70d27571775aff0e51569c` |
| Licencia | CC BY 4.0 |
| Fecha de consulta | 2026-10-25 |
| Muestra | No. Se usa el dataset completo |
| Naturaleza de los datos | **Ficticios**: la ficha indica que provienen de una compania de telecomunicaciones ficticia creada por IBM Sample Datasets |

## 0. Preparacion del entorno
Se ubica la carpeta `AE2` (que contiene `src/`) para importar las clases propias y definir rutas relativas, sin importar desde que carpeta se abra el notebook.

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")  # oculta avisos de tqdm/ipywidgets que no afectan el resultado

import pandas as pd
from datasets import disable_progress_bars
from huggingface_hub.utils import logging as hf_logging

disable_progress_bars()
hf_logging.set_verbosity_error()  # el dataset es publico: no requiere HF_TOKEN

# Buscar la carpeta AE2 subiendo desde el directorio actual
AE2_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "src" / "hf_dataset_loader.py").exists()
               or (p / "AE2" / "src").exists())
if (AE2_DIR / "AE2").exists():
    AE2_DIR = AE2_DIR / "AE2"
sys.path.insert(0, str(AE2_DIR))

from src.hf_dataset_loader import HFDatasetLoader
from src.dataset_summarizer import DatasetSummarizer

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print("Carpeta AE2:", AE2_DIR)
print("pandas", pd.__version__)

Carpeta AE2: C:\Users\diego.bedoya\orca\proyecto_integrador\AE2
pandas 2.0.3


## 1. Obtencion del dataset desde Hugging Face
Se usa la clase `HFDatasetLoader` (`src/hf_dataset_loader.py`), que llama a `datasets.load_dataset` y convierte el resultado a un DataFrame de pandas con `to_pandas()`. Se fija la **revision** del repositorio para que la descarga sea reproducible. El dataset es publico: no requiere token.

In [2]:
DATASET_ID = "scikit-learn/churn-prediction"
SPLIT = "train"
CONFIG = None  # el dataset solo tiene la configuracion por defecto
REVISION = "aa09900373d90780ee70d27571775aff0e51569c"

loader = HFDatasetLoader(DATASET_ID, split=SPLIT, config_name=CONFIG, revision=REVISION)
print(loader)
print("Splits disponibles:", loader.available_splits())

df = loader.load()
print("Tipo de objeto:", type(df))

HFDatasetLoader(dataset_id='scikit-learn/churn-prediction', split='train', config_name=None, revision='aa09900373d90780ee70d27571775aff0e51569c')


Splits disponibles: ['train']


Tipo de objeto: <class 'pandas.core.frame.DataFrame'>


## 2. Descripcion basica del DataFrame
### 2.1 Algunos registros

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.sample(5, random_state=42)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
185,1024-GUALD,Female,0,Yes,No,1,No,No phone service,DSL,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,24.80,24.8,Yes
2715,0484-JPBRU,Male,0,No,No,41,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),25.25,996.45,No
3825,3620-EHIMZ,Female,0,Yes,Yes,52,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.35,1031.7,No
1807,6910-HADCM,Female,0,No,No,1,Yes,No,Fiber optic,No,No,Yes,No,No,No,Month-to-month,No,Electronic check,76.35,76.35,Yes
132,8587-XYZSF,Male,0,No,No,67,Yes,No,DSL,No,No,No,Yes,No,No,Two year,No,Bank transfer (automatic),50.55,3260.1,No


### 2.2 Dimensiones (filas y columnas)

In [5]:
filas, columnas = df.shape
print(f"Filas: {filas}")
print(f"Columnas: {columnas}")

Filas: 7043
Columnas: 21


### 2.3 Nombres y tipos de las columnas

In [6]:
columnas_info = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "valores_unicos": df.nunique(),
    "ejemplo": df.iloc[0],
})
columnas_info

,tipo,valores_unicos,ejemplo
customerID,object,7043,7590-VHVEG
gender,object,2,Female
SeniorCitizen,int64,2,0
Partner,object,2,Yes
Dependents,object,2,No
tenure,int64,73,1
PhoneService,object,2,No
MultipleLines,object,3,No phone service
InternetService,object,3,DSL
OnlineSecurity,object,3,No


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


### 2.4 Valores nulos por columna
Ademas de `isnull()`, se cuentan los textos vacios (`' '`), porque pandas no los reconoce como nulos.

In [8]:
nulos = pd.DataFrame({
    "nulos": df.isnull().sum(),
    "textos_vacios": df.apply(lambda c: c.astype(str).str.strip().eq("").sum() if c.dtype == object else 0),
})
nulos

,nulos,textos_vacios
customerID,0,0
gender,0,0
SeniorCitizen,0,0
Partner,0,0
Dependents,0,0
tenure,0,0
PhoneService,0,0
MultipleLines,0,0
InternetService,0,0
OnlineSecurity,0,0


In [9]:
# Registros con TotalCharges vacio: revisar su antiguedad
df.loc[df["TotalCharges"].str.strip() == "", ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


**Observacion:** el DataFrame no tiene nulos explicitos, pero `TotalCharges` llega como texto (`object`) y tiene **11 registros vacios**. Todos corresponden a clientes con `tenure = 0`, es decir, clientes recien vinculados que aun no tienen cargos acumulados. En la siguiente etapa se convertira la columna a numero y se decidira como tratar esos 11 casos. En esta etapa **no se modifica** el DataFrame.

### 2.5 Valores posibles de las variables categoricas
Permite construir el diccionario inicial de variables del documento.

In [10]:
categoricas = [c for c in df.columns if df[c].dtype == object and c not in ("customerID", "TotalCharges")]
pd.DataFrame({
    "categorias": [", ".join(sorted(df[c].unique())) for c in categoricas],
    "n_categorias": [df[c].nunique() for c in categoricas],
}, index=categoricas)

,categorias,n_categorias
gender,"Female, Male",2
Partner,"No, Yes",2
Dependents,"No, Yes",2
PhoneService,"No, Yes",2
MultipleLines,"No, No phone service, Yes",3
InternetService,"DSL, Fiber optic, No",3
OnlineSecurity,"No, No internet service, Yes",3
OnlineBackup,"No, No internet service, Yes",3
DeviceProtection,"No, No internet service, Yes",3
TechSupport,"No, No internet service, Yes",3


In [11]:
df[["SeniorCitizen", "tenure", "MonthlyCharges"]].describe().round(2)

,SeniorCitizen,tenure,MonthlyCharges
count,7043.00,7043.00,7043.00
mean,0.16,32.37,64.76
std,0.37,24.56,30.09
min,0.00,0.00,18.25
25%,0.00,9.00,35.50
50%,0.00,29.00,70.35
75%,0.00,55.00,89.85
max,1.00,72.00,118.75


## 3. Que representa cada fila
Segun la ficha del dataset, **cada fila representa un cliente** de la compania de telecomunicaciones ficticia. Se verifica que el identificador `customerID` no se repite:

In [12]:
print("customerID unico por fila:", df["customerID"].is_unique)
print("Clientes distintos:", df["customerID"].nunique(), "de", len(df), "filas")

customerID unico por fila: True


Clientes distintos: 7043 de 7043 filas


## 4. Variables que respaldan la necesidad

**Necesidad propuesta (escenario hipotetico):** el area de retencion de clientes de la compania necesita conocer **que perfiles de cliente concentran la cancelacion del servicio** para priorizar sus programas de fidelizacion.

**Pregunta de analisis:** *Entre los 7.043 clientes registrados, como varia la tasa de cancelacion (`Churn`) segun el tipo de contrato, la antiguedad, los servicios contratados, la forma de pago, el nivel de cargos y el perfil demografico, y que segmentos presentan las tasas mas altas?*

| Grupo | Variables | Papel en el analisis |
|---|---|---|
| Variable de resultado | `Churn` | Indica si el cliente se fue en el ultimo mes (Yes/No); permite calcular tasas de cancelacion |
| Cuenta / contrato | `Contract`, `PaymentMethod`, `PaperlessBilling` | Categorias para comparar tasas entre condiciones comerciales |
| Antiguedad | `tenure` | Numerica; se agrupara en rangos para comparar periodos de permanencia |
| Cargos | `MonthlyCharges`, `TotalCharges` | Numericas; comparar distribuciones entre quienes cancelan y quienes no |
| Servicios | `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | Categorias para comparar tasas segun servicios contratados |
| Perfil demografico | `gender`, `SeniorCitizen`, `Partner`, `Dependents` | Categorias para describir grupos de clientes |
| Identificador | `customerID` | Solo identifica la fila; no se analiza |

Como verificacion de que la variable de resultado esta disponible y tiene ambas categorias, se muestra su conteo (no es aun el analisis):

In [13]:
pd.DataFrame({
    "clientes": df["Churn"].value_counts(),
    "porcentaje_%": (df["Churn"].value_counts(normalize=True) * 100).round(2),
})

,clientes,porcentaje_%
Churn,,
No,5174,73.46
Yes,1869,26.54


## 5. Generacion automatica del resumen en .txt
Se genera el resumen **a partir del mismo DataFrame `df`** revisado arriba. La clase `DatasetSummarizer` crea la carpeta `resultados/` si no existe y **sobrescribe** el archivo en cada ejecucion, con codificacion UTF-8.

In [14]:
RUTA_RESUMEN = AE2_DIR / "resultados" / "resumen_dataframe.txt"

procedencia = {
    "Dataset": "Telco Customer Churn (IBM Sample Dataset)",
    "Identificador": DATASET_ID,
    "Enlace": f"https://huggingface.co/datasets/{DATASET_ID}",
    "Configuracion": CONFIG or "default",
    "Split": SPLIT,
    "Revision": REVISION,
    "Licencia": "CC BY 4.0",
    "Muestra": "No - DataFrame completo, sin filtros ni transformaciones",
    "Naturaleza": "Datos ficticios (IBM Sample Datasets)",
}

summarizer = DatasetSummarizer(df, title="Resumen del DataFrame - Telco Customer Churn",
                               source=procedencia)
ruta = summarizer.save_txt(RUTA_RESUMEN)
print("Resumen guardado en:", ruta.relative_to(AE2_DIR.parent))

Resumen guardado en: AE2\resultados\resumen_dataframe.txt


### 5.1 Verificacion del archivo generado
Se lee el archivo y se comprueba que coincide con el DataFrame (filas, columnas y nombres).

In [15]:
contenido = ruta.read_text(encoding="utf-8")

assert f"Filas: {df.shape[0]}" in contenido
assert f"Columnas: {df.shape[1]}" in contenido
assert all(col in contenido for col in df.columns)
assert DATASET_ID in contenido
assert any(l.startswith("Split") and l.endswith(SPLIT) for l in contenido.splitlines())
print("Verificacion OK: el resumen corresponde al DataFrame cargado.\n")
print(contenido)

Verificacion OK: el resumen corresponde al DataFrame cargado.

RESUMEN DEL DATAFRAME - TELCO CUSTOMER CHURN
Generado: 2026-10-25 20:08:04

1. Procedencia del dataset
--------------------------
Dataset       : Telco Customer Churn (IBM Sample Dataset)
Identificador : scikit-learn/churn-prediction
Enlace        : https://huggingface.co/datasets/scikit-learn/churn-prediction
Configuracion : default
Split         : train
Revision      : aa09900373d90780ee70d27571775aff0e51569c
Licencia      : CC BY 4.0
Muestra       : No - DataFrame completo, sin filtros ni transformaciones
Naturaleza    : Datos ficticios (IBM Sample Datasets)

2. Dimensiones
--------------
Filas: 7043
Columnas: 21

3. Columnas y tipos de datos
----------------------------
 n          columna    tipo  valores_unicos
 1       customerID  object            7043
 2           gender  object               2
 3    SeniorCitizen   int64               2
 4          Partner  object               2
 5       Dependents  object       

## 6. Conclusiones de la etapa 1

- Se obtuvo el dataset `scikit-learn/churn-prediction` (split `train`, revision fijada) desde Hugging Face y se cargo en un DataFrame de **7.043 filas y 21 columnas**; se usa completo, sin muestreo.
- Cada fila es un **cliente** identificado por `customerID` (unico).
- Hay 18 variables categoricas (texto), 2 numericas enteras (`SeniorCitizen`, `tenure`), 1 decimal (`MonthlyCharges`) y `TotalCharges`, que es numerica pero llega como texto con 11 valores vacios.
- La variable `Churn` y las variables de contrato, servicios, cargos y perfil permiten responder una pregunta **descriptiva** de comparacion de tasas de cancelacion entre segmentos.
- Los datos son **ficticios**: los resultados serviran para un escenario hipotetico y no describen a una empresa real.
- El resumen se genero en `AE2/resultados/resumen_dataframe.txt`.